# Splitting the data into training, test, and validation set

In this script, we split the data into a training, validation and test set. The validation set could be used to compare different models (although this is not relevant here). We make the divide by year to minimize leakage.

In [12]:
import pandas as pd
import numpy as np
import random
import math

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
path_to_data = '/content/drive/MyDrive/TL/data/'
df_party = pd.read_parquet(path_to_data + 'speeches_prep_party.parquet')
df_noparty = pd.read_parquet(path_to_data + 'speeches_prep_party.parquet')

## Divide data frames with party names into training, test and validation data:



In [15]:
df_party_test = df_party[df_party['date'].str.contains('2024|2025')]
df_party_val = df_party[df_party['date'].str.contains('2023')]
df_party_train = df_party[df_party['date'].str.contains('2020|2021|2022')]

Check how many samples they each have:

In [16]:
print(df_party_test.shape)
print(df_party_val.shape)
print(df_party_train.shape)

(12568, 12)
(14717, 12)
(37797, 12)


Check how balanced they are:

In [17]:
df_party_train.groupby('party').describe()

labels                                    no_words              \
             count mean  std  min  25%  50%  75%  max    count        mean   
party                                                                        
AfD         6403.0  5.0  0.0  5.0  5.0  5.0  5.0  5.0   6403.0  385.246135   
CDU/CSU     6706.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   6706.0  381.511482   
Die Grünen  5682.0  2.0  0.0  2.0  2.0  2.0  2.0  2.0   5682.0  385.254840   
Die Linke   7119.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0   7119.0  379.324343   
FDP         6083.0  4.0  0.0  4.0  4.0  4.0  4.0  4.0   6083.0  387.386158   
SPD         5804.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   5804.0  385.782219   

            ... no_words_stripped         no_chars                           \
            ...               75%     max    count         mean         std   
party       ...                                                               
AfD         ...            655.00  1771.0   6403.0  2713.326409  343.211346   
CDU/CSU     ...            824.75  4841.0   6706.0  2620.194900  469.329111   
Die Grünen  ...            714.00  2862.0   5682.0  2662.435938  384.737192   
Die Linke   ...            608.00  2490.0   7119.0  2642.832280  370.842695   
FDP         ...            688.00  4521.0   6083.0  2679.958080  373.233530   
SPD         ...            842.00  9345.0   5804.0  2660.099242  442.873245   

                                                     
               min      25%     50%     75%     max  
party                                                
AfD         1241.0  2684.00  2834.0  2919.0  2999.0  
CDU/CSU     1252.0  2465.25  2857.0  2939.0  2998.0  
Die Grünen  1198.0  2582.00  2815.0  2915.0  3000.0  
Die Linke   1254.0  2550.00  2786.0  2893.0  3000.0  
FDP         1211.0  2634.00  2823.0  2912.0  2998.0  
SPD         1237.0  2629.75  2860.0  2939.0  3000.0  

[6 rows x 32 columns]

In [18]:
df_party_test.groupby('party').describe()

labels                                    no_words              \
             count mean  std  min  25%  50%  75%  max    count        mean   
party                                                                        
AfD         2140.0  5.0  0.0  5.0  5.0  5.0  5.0  5.0   2140.0  379.396729   
CDU/CSU     1968.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   1968.0  380.041159   
Die Grünen  2543.0  2.0  0.0  2.0  2.0  2.0  2.0  2.0   2543.0  387.948879   
Die Linke   1319.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0   1319.0  331.961334   
FDP         2225.0  4.0  0.0  4.0  4.0  4.0  4.0  4.0   2225.0  385.653034   
SPD         2373.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   2373.0  386.175727   

            ... no_words_stripped         no_chars                           \
            ...               75%     max    count         mean         std   
party       ...                                                               
AfD         ...             644.0  1351.0   2140.0  2655.708879  389.068378   
CDU/CSU     ...             747.0  2890.0   1968.0  2611.503557  477.880018   
Die Grünen  ...             775.0  2950.0   2543.0  2660.823437  391.354222   
Die Linke   ...             420.5  1512.0   1319.0  2314.374526  458.748554   
FDP         ...             741.0  4892.0   2225.0  2670.694831  392.398213   
SPD         ...             808.0  4195.0   2373.0  2655.365782  444.948783   

                                                     
               min      25%     50%     75%     max  
party                                                
AfD         1192.0  2569.50  2808.0  2915.0  2998.0  
CDU/CSU     1270.0  2450.75  2849.5  2939.0  2998.0  
Die Grünen  1237.0  2597.00  2817.0  2910.0  2998.0  
Die Linke   1312.0  1902.50  2361.0  2746.0  2999.0  
FDP         1241.0  2627.00  2836.0  2918.0  2998.0  
SPD         1237.0  2616.00  2857.0  2936.0  2998.0  

[6 rows x 32 columns]

In [19]:
df_party_val.groupby('party').describe()

labels                                    no_words              \
             count mean  std  min  25%  50%  75%  max    count        mean   
party                                                                        
AfD         2304.0  5.0  0.0  5.0  5.0  5.0  5.0  5.0   2304.0  382.687934   
CDU/CSU     2173.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   2173.0  381.713300   
Die Grünen  2622.0  2.0  0.0  2.0  2.0  2.0  2.0  2.0   2622.0  386.017544   
Die Linke   2409.0  3.0  0.0  3.0  3.0  3.0  3.0  3.0   2409.0  364.559153   
FDP         2539.0  4.0  0.0  4.0  4.0  4.0  4.0  4.0   2539.0  384.667192   
SPD         2670.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0   2670.0  383.694007   

            ... no_words_stripped         no_chars                           \
            ...               75%     max    count         mean         std   
party       ...                                                               
AfD         ...             625.0  1817.0   2304.0  2686.638455  343.589824   
CDU/CSU     ...             744.0  2181.0   2173.0  2625.599172  456.100229   
Die Grünen  ...             759.0  2690.0   2622.0  2668.331045  382.445308   
Die Linke   ...             569.0  1539.0   2409.0  2549.299709  433.454158   
FDP         ...             730.0  5241.0   2539.0  2662.285939  419.828297   
SPD         ...             800.0  4012.0   2670.0  2656.122846  451.333130   

                                                      
               min      25%     50%      75%     max  
party                                                 
AfD         1258.0  2595.75  2821.0  2915.00  3000.0  
CDU/CSU     1266.0  2487.00  2845.0  2936.00  3000.0  
Die Grünen  1257.0  2585.00  2823.0  2920.75  2998.0  
Die Linke   1218.0  2366.00  2719.0  2879.00  2998.0  
FDP         1288.0  2644.00  2840.0  2920.50  2998.0  
SPD         1177.0  2625.50  2864.0  2941.00  3000.0  

[6 rows x 32 columns]

They are not super balanced, but it looks okay for now. If we wanted to, we could decide to use balanced subsets of these data frames for training, but the imbalance is not as bad as before.

## Divide data frames *without* party names into training, test and validation data:


In [20]:
df_noparty_test = df_noparty[df_noparty['date'].str.contains('2024|2025')]
df_noparty_val = df_noparty[df_noparty['date'].str.contains('2023')]
df_noparty_train = df_noparty[df_noparty['date'].str.contains('2020|2021|2022')]

In [21]:
print(df_noparty_test.shape)
print(df_noparty_val.shape)
print(df_noparty_train.shape)

(12568, 12)
(14717, 12)
(37797, 12)


Save the results:

In [22]:
df_party_test.to_parquet(path_to_data + 'speeches_party_test.parquet')
df_party_val.to_parquet(path_to_data + 'speeches_party_val.parquet')
df_party_train.to_parquet(path_to_data + 'speeches_party_train.parquet')

df_noparty_test.to_parquet(path_to_data + 'speeches_noparty_test.parquet')
df_noparty_val.to_parquet(path_to_data + 'speeches_noparty_val.parquet')
df_noparty_train.to_parquet(path_to_data + 'speeches_noparty_train.parquet')